In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean & std
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=1024, shuffle=False)

In [ ]:
import torch.nn as nn
import torch.optim as optim


# Using PyTorch's built-in methods
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.filter_size_1 = 3
        self.filter_size_2 = 2
        self.filter_size_3 = 2
        self.num_filters_1 = 64
        self.num_filters_2 = 32
        self.padding = 0
        self.stride = 1

        # device, dtype = X_train.device, torch.float32
        h_in, w_in = int(X_train.shape[1]), int(X_train.shape[2])

        h_conv_1 = (h_in + 2 * self.padding - self.filter_size_1) // self.stride + 1
        w_conv_1 = (w_in + 2 * self.padding - self.filter_size_1) // self.stride + 1
        h_pool_1 = h_conv_1 // 2
        w_pool_1 = w_conv_1 // 2

        h_conv_2 = (h_pool_1 + 2 * self.padding - self.filter_size_2) // self.stride + 1
        w_conv_2 = (w_pool_1 + 2 * self.padding - self.filter_size_2) // self.stride + 1
        h_pool_2 = h_conv_2 // 2
        w_pool_2 = w_conv_2 // 2

        flat_dim = self.num_filters_2 * h_pool_2 * w_pool_2

        self.conv_block = nn.Sequential(
            nn.Conv2d(
                1,
                self.num_filters_1,
                kernel_size=self.filter_size_1,
                padding=self.padding,
                stride=self.stride,
            ),  # 28x28 -> 26x26
            nn.ReLU(),
            nn.MaxPool2d(2),  # 26x26 → 13x13
            nn.Conv2d(
                self.num_filters_1,
                self.num_filters_2,
                kernel_size=self.filter_size_2,
                padding=self.padding,
                stride=self.stride,
            ),  # 13x13 -> 12x12
            nn.ReLU(),
            nn.MaxPool2d(2),  # 12x12 → 6x6
        )
        self.l_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_dim, 128),
            nn.ReLU(),
            # nn.Dropout(0.5),
            nn.Linear(128, 10),
        )

    def forward(self, Xs):
        return self.l_block(self.conv_block(Xs))

In [ ]:
# Training the model
device = X_train.device
model  = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train(epochs=5):
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

    print(f"Epoch {epoch} | Loss: {total_loss / len(train_loader):.4f}")

train()


In [ ]:
def evaluate():
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()

    print(f"Train Accuracy: {correct / len(train_dataset) * 100:.2f}%")
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()

    print(f"Test Accuracy: {correct / len(test_dataset) * 100:.2f}%")


evaluate()